# ASTraM Event Forecasting - Data Cleaning & EDA

This notebook performs the initial data ingestion, cleaning, and exploratory data analysis for the ASTraM event dataset.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

# Ensure data directory exists for outputs
os.makedirs('../data', exist_ok=True)

## 1. Data Ingestion

In [2]:
raw_file_path = r'../Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
df = pd.read_csv(raw_file_path, on_bad_lines='skip')
print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (8173, 46)


## 2. Drop Dead Columns
As identified in the audit, these columns are entirely missing or too sparse to be useful (e.g., resource allocations).

In [3]:
dead_columns = ['meta_data', 'comment', 'map_file', 'assigned_to_police_id']
df = df.drop(columns=[c for c in dead_columns if c in df.columns])
print(f"Dataset shape after dropping dead columns: {df.shape}")

Dataset shape after dropping dead columns: (8173, 42)


## 3. Timezone Conversion (UTC to IST)
The timestamps are currently in UTC. We need them in IST (+05:30) to properly extract hour of day and day of week features later.

In [4]:
dt_cols = ['start_datetime', 'end_datetime', 'closed_datetime']

for col in dt_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    # Convert to IST
    df[col] = df[col].dt.tz_convert('Asia/Kolkata') if df[col].dt.tz else df[col].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')

## 4. Target Construction & Cleaning
We need to combine `end_datetime` (used for planned events) and `closed_datetime` (used for unplanned events) to get the true resolution time.

In [5]:
df['resolution_dt'] = df['end_datetime'].fillna(df['closed_datetime'])

# Compute duration in minutes
df['duration_minutes'] = (df['resolution_dt'] - df['start_datetime']).dt.total_seconds() / 60.0

# 1. Drop active incidents (censored data)
df = df[df['status'] != 'active'].copy()

# 2. Drop negative durations (data entry errors)
df = df[(df['duration_minutes'] >= 0) | df['duration_minutes'].isna()].copy()

# 3. Winsorize extreme outliers (cap at 48 hours / 2880 minutes)
MAX_DURATION = 48 * 60
df['duration_minutes'] = df['duration_minutes'].clip(upper=MAX_DURATION)

print(f"Final shape after basic cleaning: {df.shape}")
print(f"Rows with valid duration labels: {df['duration_minutes'].notna().sum()}")

Final shape after basic cleaning: (7116, 44)
Rows with valid duration labels: 3428


## 5. Exploratory Data Analysis

In [6]:
# Top event causes
cause_counts = df['event_cause'].value_counts().reset_index()
cause_counts.columns = ['event_cause', 'count']
fig = px.bar(cause_counts, x='event_cause', y='count', title='Incident Counts by Cause')
fig.show()

# Priority vs Closure Probability
closure_prob = df.groupby('priority')['requires_road_closure'].mean().reset_index()
fig2 = px.bar(closure_prob, x='priority', y='requires_road_closure', title='Probability of Road Closure by Priority')
fig2.show()

# Median Duration by Cause
median_duration = df.dropna(subset=['duration_minutes']).groupby('event_cause')['duration_minutes'].median().sort_values().reset_index()
fig3 = px.bar(median_duration, x='event_cause', y='duration_minutes', title='Median Duration (Minutes) by Cause')
fig3.show()

## 6. Export Cleaned Data

In [7]:
df.to_csv('../data/cleaned_astram_events.csv', index=False)
print("Saved cleaned dataset to data/cleaned_astram_events.csv")

Saved cleaned dataset to data/cleaned_astram_events.csv
